# img_det baseline - Colab (GPU)

Scores the published `aludam` 1.0.0 image detector on **CommunityForensics-Eval**
and produces **`result.txt`** (auto-downloaded at the end).

1. *Runtime > Change runtime type > GPU* (CPU also works - just slower).
2. *Runtime > Run All*.
3. Hand me the downloaded `result.txt`.

First run installs `aludam`, clones this repo (or lets you upload
`img_eval_core.py` if not pushed yet), fetches the 665 MB checkpoint and
~1,500 test images. Everything is resumable - re-run is cheap.

Stamped: general-deepfake distribution, **not** validated for government-ID
/ KYC scans. No training, no package changes.

In [ ]:
import importlib.util
NEED = {"numpy": "numpy", "pandas": "pandas", "PIL": "Pillow",
        "cv2": "opencv-python", "pyarrow": "pyarrow", "fsspec": "fsspec",
        "requests": "requests", "aiohttp": "aiohttp",
        "sklearn": "scikit-learn", "torch": "torch",
        "transformers": "transformers", "aludam": "aludam"}
missing = [p for m, p in NEED.items() if importlib.util.find_spec(m) is None]

pkgs = []
if "aludam" in missing:
    pkgs.append("aludam[neural]")
pkgs += [p for m, p in NEED.items() if m in missing and m != "aludam"]
if pkgs:
    import subprocess
    import sys
    print("installing:", pkgs)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
still = [p for m, p in NEED.items() if importlib.util.find_spec(m) is None]
assert not still, "still missing: %s" % still

import aludam
import torch
print("aludam", aludam.__version__, "| torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path("/content/Deep-Fake-Analysis")
if not (REPO / "aludam").is_dir():
    print("cloning repo ...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Ashutosh3021/Deep-Fake-Analysis",
         str(REPO)],
        check=True, cwd="/content")
os.environ["ALUDAM_REPO"] = str(REPO)
os.environ["ALUDAM_MODELS_DIR"] = str(REPO / "models")
print("repo:", REPO)

In [ ]:
import sys

sys.path.insert(0, str(REPO / "notebooks"))
try:
    import img_eval_core as core
except ImportError:
    # notebooks/ may not be on GitHub yet (not pushed) - upload the file.
    target = REPO / "notebooks" / "img_eval_core.py"
    if not target.exists():
        print("img_eval_core.py not found in clone - upload it when prompted")
        from google.colab import files
        uploaded = files.upload()
        data = uploaded.get("img_eval_core.py")
        if data is None:
            raise SystemExit("please upload notebooks/img_eval_core.py")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    sys.path.insert(0, str(REPO / "notebooks"))
    import img_eval_core as core
print("core loaded from:", core.__file__)

In [ ]:
CONFIG = dict(
    per_class=750,     # rows per class (plan default 750)
    seed=20261004,     # selection + split seed
    device="auto",     # "auto" = CUDA when available, else CPU
    workers=8,         # parallel metadata readers (HF-friendly)
    score_every=25,    # progress log interval, in images
)
print(CONFIG)

In [ ]:
# Step 1: metadata-only index of the dataset (payload column never read).
index_rows = core.build_index(REPO, workers=CONFIG["workers"])
print("indexed rows:", len(index_rows))

In [ ]:
# Step 2: label evidence + source-grouped dev/cal/test split.
selected, splits, evidence, warns = core.select_and_split(
    index_rows, per_class=CONFIG["per_class"], seed=CONFIG["seed"])
print("selected:", len(selected), "| warnings:", len(warns))

In [ ]:
# Step 3: download ONLY the selected images (resumable; re-run skips done).
PAYLOAD_DIR = REPO / "aludam" / "eval" / "data" / "index" / "img_payload"
payload_stats = core.fetch_payload(selected, PAYLOAD_DIR)
payload_stats

In [ ]:
# Step 4: load aludam img_det (weights are fetched once if missing).
detector, device_used, device_notes = core.get_detector(CONFIG["device"])
print("device:", device_used)
for n in device_notes:
    print(" -", n)

In [ ]:
# Step 5: score every selected image (resumable JSONL; re-run skips done).
SCORES_PATH = REPO / "aludam" / "eval" / "data" / "index" / "img_scores.jsonl"
score_stats = core.score_all(detector, selected, PAYLOAD_DIR, SCORES_PATH,
                             every=CONFIG["score_every"])
score_stats

In [ ]:
# Step 6: metrics -> per-signal diagnosis -> dual-budget fit -> test verify.
OUT_PATH = Path("/content/result.txt")
text = core.analyze_and_write(
    repo=REPO, selected=selected, splits=splits, evidence=evidence,
    warnings_=warns, payload_stats=payload_stats, score_stats=score_stats,
    scores_path=SCORES_PATH, out_path=OUT_PATH,
    per_class=CONFIG["per_class"], seed=CONFIG["seed"],
    device_used=device_used, device_notes=device_notes)
print(text)
print("RESULT FILE:", OUT_PATH)

In [ ]:
from google.colab import files
files.download(str(Path("/content/result.txt")))

## Done

Send me the downloaded `result.txt`. If anything failed mid-way, just
*Run All* again - index, downloads and scores all resume.